# Setup

This notebook converts strongly labelled data into the info tables required for training and testing Voxaboxen models.

In [32]:
import json
import pandas as pd
import os
from urllib.request import urlretrieve
from pydub import AudioSegment
from tqdm import tqdm

# Step 1 - Create Selection Tables

The selection tables are .txt files with only the following columns required: ``Begin Time (s)``, ``End Time (s)``, ``Annotation``.
They can be created through one of the following options:
* Option A: Parsing a single JSON file into multiple Raven selection tables
* Option B: Formatting existing Raven selection tables

Only run the cells under either Option A or B, before moving onto Step 2.

##  Option A: Parse JSON File

Follow these steps if strongly labelled data has been exported from Soundbox as a JSON file.

In [ ]:
JSON_PATH = '/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lit_peronii/Lit_peronii_soundbox_labels.json'

with open(JSON_PATH, 'r') as file:
    data = json.load(file)

In [29]:
# Map recording uuid to capture ID
recordings = data['recordings']
recordings_df = pd.DataFrame.from_dict(recordings)
uuid_to_capture_map = recordings_df.set_index('uuid')['path'].to_dict()
uuid_to_capture_map = {k: v.split('.')[0] for k,v in uuid_to_capture_map.items()}

In [30]:
# Extract start/end times from sound events
sound_events = data['sound_events']
sound_events_df = pd.DataFrame.from_dict(sound_events)
try:
    sound_events_df["begin_time"] = sound_events_df["geometry"].apply(lambda x: x["coordinates"][0])
    sound_events_df["end_time"] = sound_events_df["geometry"].apply(lambda x: x["coordinates"][2])
except Exception:
    sound_events_df["begin_time"] = sound_events_df["geometry"].apply(lambda x: eval(x)["coordinates"][0])
    sound_events_df["end_time"] = sound_events_df["geometry"].apply(lambda x: eval(x)["coordinates"][2])

sound_events_df

,id,uuid,recording_id,geometry_type,geometry,created_on,recording,begin_time,end_time
0,2273,607de690-5398-40f4-97cd-c50943379ea6,3858,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [1.1544...",2025-08-20 23:59:09.767302,5d4c7161-1663-4ce9-ac63-20875c69c670,1.154435,2.813926
1,2274,f669ba8e-e544-484e-9cea-1d2d93c4d3d5,3858,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [9.9670...",2025-08-20 23:59:30.492477,5d4c7161-1663-4ce9-ac63-20875c69c670,9.967009,11.255600
2,2275,d9ab64b5-d6b4-4e91-a457-66ace60550bd,3858,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [24.385...",2025-08-20 23:59:55.569458,5d4c7161-1663-4ce9-ac63-20875c69c670,24.385375,26.239876
3,2276,6c19c546-9bfd-45a7-a94d-545452fa0cdb,3858,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [27.802...",2025-08-21 00:00:09.038933,5d4c7161-1663-4ce9-ac63-20875c69c670,27.802931,29.141230
4,2277,843c26a6-ce30-4812-93a4-c09ff3c3516a,3822,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [0.0527...",2025-08-21 00:05:18.122377,9d346f11-4175-46de-86aa-90bc08aed1d3,0.052702,1.269952
...,...,...,...,...,...,...,...,...,...
524,10311,61ffa873-eeb2-41f8-94bd-8112e4d528db,2501,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [30.020...",2025-09-04 06:38:07.129709,8079fc72-d476-4759-a7f7-b71c386d011e,30.020008,32.036523
525,10312,8100326b-d362-4f8d-95a3-bc88291856d4,2501,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [36.692...",2025-09-04 06:38:30.533382,8079fc72-d476-4759-a7f7-b71c386d011e,36.692139,38.492390
526,10313,e33a553d-0d90-4e21-9020-77531ebe8df4,2501,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [42.753...",2025-09-04 06:38:48.298642,8079fc72-d476-4759-a7f7-b71c386d011e,42.753052,44.483163
527,10314,6aa5b55a-c99a-41ca-bea4-f7aa99cf41d5,2501,BoundingBox,"{'type': 'BoundingBox', 'coordinates': [51.346...",2025-09-04 06:39:03.858649,8079fc72-d476-4759-a7f7-b71c386d011e,51.346414,53.111596


In [ ]:
# Convert to Raven selection tables
OUTPUT_DIR = "/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables"


selection_table_dir = OUTPUT_DIR
os.makedirs(selection_table_dir, exist_ok=True)

count_skipped = 0
for uuid, capture_id in uuid_to_capture_map.items():

    # Find sound events corresponding to this recording
    sound_events = sound_events_df[sound_events_df['recording'] == uuid]

    if len(sound_events) == 0:
        count_skipped += 1
        continue
    
    output_path = os.path.join(selection_table_dir, f"{capture_id}_selection_table.txt")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("Begin Time (s)\tEnd Time (s)\tAnnotation\n")

    for _, row in sound_events.iterrows():
        with open(output_path, "a", encoding="utf-8") as f:
            f.write(f"{row['begin_time']}\t{row['end_time']}\tevent\n")

print(f"Finished writing {len(uuid_to_capture_map) - count_skipped} selection tables to: {selection_table_dir}")

## Option B: Raw Raven Export

Follow these steps if strongly labelled data has been exported directly off Raven.

In [35]:
# Define input directory containing Raven selection tables
INPUT_DIR = "/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables"


selection_table_dir = INPUT_DIR
template_suffix = ".Table.1.selections.txt"

for filename in os.listdir(selection_table_dir):
    if filename.endswith(template_suffix):
        
        full_path = os.path.join(selection_table_dir, filename)
        print(f"Processing: {filename}")

        try:
            # Load Raven selection table (tab-separated)
            df = pd.read_csv(full_path, sep="\t")

            # Keep only required columns
            df_clean = df[["Begin Time (s)", "End Time (s)"]].copy()

            # Remove duplicate rows (waveform/spectrogram duplicates)
            df_clean = df_clean.drop_duplicates()

            # Add Annotation column
            df_clean["Annotation"] = "event"

            # Save cleaned file
            output_name = filename.replace(template_suffix, "_selection_table.txt")
            output_path = os.path.join(selection_table_dir, output_name)

            df_clean.to_csv(output_path, sep="\t", index=False)

            print(f"Saved: {output_path}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Step 2 - Create Info Table
The info table is a CSV file with 3 columns:

* ``fn``: Unique filename associated with each audio file
* ``audio_fp``: Filepaths to audio files in train set
* ``selection_table_fp``: Filepath to Raven selection tables

In [ ]:
# Load captures table
captures_path = "/home/admin/julia-dev/data/frogid_captures/data_tables/FrogID_Captures_2025-03-09T21-01-13+0000.csv"
captures_df = pd.read_csv(captures_path)

/tmp/ipykernel_193118/3616435119.py:3: DtypeWarning: Columns (32,34,35,39) have mixed types. Specify dtype option on import or set low_memory=False.
  captures_df = pd.read_csv(captures_path)


In [ ]:
# Download audio files
AUDIO_DIR = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lit_peronii/audio"


def download_audio(audio_dir, capture_id, audio_url):
    save_path = os.path.join(audio_dir, f'{capture_id}.aac')
    urlretrieve(audio_url, save_path)
    audio = AudioSegment.from_file(save_path)
    wav_path = os.path.join(audio_dir, f'{capture_id}.wav')
    audio.export(wav_path, format='wav')
    os.remove(save_path)


captures_labelled = [x.split('_')[0] for x in os.listdir(selection_table_dir)]
for capture_id in tqdm(captures_labelled, desc="Downloading audio"):
    row = captures_df[captures_df['id'] == int(capture_id)]
    audio_url = row['call_audio'].values[0]
    download_audio(AUDIO_DIR, capture_id, audio_url)

http://res.cloudinary.com/ausmus/video/upload/v1581159945/rcvfennzrhlgz1mkgx1s.aac
http://res.cloudinary.com/ausmus/video/upload/v1571911123/qsy9mniboa52htxylzso.aac
http://res.cloudinary.com/ausmus/video/upload/v1541928122/ogtoluwhfrur1rhyhczw.aac
http://res.cloudinary.com/ausmus/video/upload/v1614968087/jqgecpin7mwfnpm19wzq.aac
http://res.cloudinary.com/ausmus/video/upload/v1601691590/yjqfmjraofrlu9b0yfty.aac
http://res.cloudinary.com/ausmus/video/upload/v1608807516/wxrsbrfg1vnq67jzy3mf.aac
http://res.cloudinary.com/ausmus/video/upload/v1579260019/wgslhbzi0goanlg27c6b.aac
http://res.cloudinary.com/ausmus/video/upload/v1564996876/xwkjvfw99y25gbbvjkjw.aac
http://res.cloudinary.com/ausmus/video/upload/v1634128090/ojufmlmpxaesssyuhzb8.aac
http://res.cloudinary.com/ausmus/video/upload/v1611226752/gmtkqb1n1hkosdwobd4w.aac
http://res.cloudinary.com/ausmus/video/upload/v1541237925/bxskamjpo6tdcvjtwu3x.aac
http://res.cloudinary.com/ausmus/video/upload/v1542706952/w9udi7a79fmkchaduz3a.aac
http

In [ ]:
CSV_OUT = 'datasets/frogid/Lit_peronii_info.csv'


# Generate pandas dataframe
info_table = []
for capture_id in captures_labelled:
    info_table.append({
        'fn': capture_id,
        'audio_fp': os.path.join(AUDIO_DIR, f'{capture_id}.wav'),
        'selection_table_fp': os.path.join(selection_table_dir, f'{capture_id}_selection_table.txt')
    })

info_df = pd.DataFrame.from_dict(info_table)

# Export as CSV
info_df.to_csv(CSV_OUT)
print(f'Saved info table to: {CSV_OUT}')

# Step 3 - Split into Train/Test/Val

In [6]:
INPUT_CSV = 'datasets/frogid/Lim_peronii_info.csv'
OUTPUT_DIR = '/home/admin/julia-dev/voxaboxen/datasets/frogid/Lim_peronii_formatted'
RANDOM_SEED = 42

TRAIN_RATIO = 0.80
TEST_RATIO  = 0.10
VAL_RATIO   = 1.0 - (TRAIN_RATIO + TEST_RATIO)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load CSV and treat first column as index
df = pd.read_csv(INPUT_CSV, index_col=0)
df.index.name = "index"

# Shuffle WITHOUT resetting index
df_shuf = df.sample(frac=1.0, random_state=RANDOM_SEED)

# Split
n = len(df_shuf)
n_train = int(n * TRAIN_RATIO)
n_test  = int(n * TEST_RATIO)

train_df = df_shuf.iloc[:n_train]
test_df  = df_shuf.iloc[n_train:n_train + n_test]
val_df   = df_shuf.iloc[n_train + n_test:]

# Write CSVs — index preserved, no extra column
train_df.to_csv(os.path.join(OUTPUT_DIR, "train_info.csv"),
                index=True, index_label=None)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_info.csv"),
               index=True, index_label=None)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_info.csv"),
              index=True, index_label=None)

print(f"Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")


Split sizes: train=41, val=6, test=5


# [Optional] Step 4 - Download Additional Audio for Inference

This optional step is for creating a test info table from a separate audio dataset, which will be used for running inference (NOT training). 

In [ ]:
# Define data source and destination
CSV_PATH = "/home/admin/julia-dev/data/frogid_captures/data_tables/fraser_cuecounts_all_data.csv"
AUDIO_DIR = "/home/admin/julia-dev/data/frogid_captures/individual_count"

df = pd.read_csv(CSV_PATH)
count = 0

for _, row in df.iterrows():
    
    capture_id = row['id']
    species = row['species']

    audio_subdir = os.path.join(AUDIO_DIR, '_'.join(species.split()).lower())
    audio_path = os.path.join(audio_subdir, f'capture_{capture_id}.wav')

    if os.path.exists(audio_path):
        continue
    else:
        try:
            url = captures_df[captures_df['id'] == capture_id]['call_audio'].values[0]
        except IndexError:
            print(f'{capture_id} not found in captures table --> SKIPPING')
        else:
            download_audio(audio_subdir, capture_id, url)

/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_caerulea_multispecies_test


In [ ]:
# Group by species for separate test directories
species_dfs = {
    species: group.copy()
    for species, group in df.groupby("species")
}

{'Limnodynastes peronii':       Unnamed: 0      id                species        lat         lng  \
 1029           1    8458  Limnodynastes peronii -34.438900  150.469000   
 1030           2    8988  Limnodynastes peronii -33.804395  151.173691   
 1031           3    9346  Limnodynastes peronii -33.223300  151.504000   
 1032           4    9581  Limnodynastes peronii -33.725997  151.152149   
 1033           5   10032  Limnodynastes peronii -33.083200  151.434000   
 ...          ...     ...                    ...        ...         ...   
 1480         452  795854  Limnodynastes peronii -36.814200  149.794000   
 1481         453  797510  Limnodynastes peronii -33.805396  151.191289   
 1482         454  799406  Limnodynastes peronii -33.721400  150.700000   
 1483         455  801041  Limnodynastes peronii -33.803944  151.189016   
 1484         456  803310  Limnodynastes peronii -32.987000  151.586000   
 
           year  eventhour  earcount  callcount   duration       temp  \


In [ ]:
# Create test info table for each species
DATASET_DIR = "/home/admin/julia-dev/voxaboxen/datasets/frogid"

def make_audio_fp(species, id_):
    folder = '_'.join(str(species).split()).lower()
    return os.path.join(AUDIO_DIR, folder, f"capture_{id_}.wav")

for species, df in species_dfs.items():
    new_df = pd.DataFrame({
        "fn": df["id"]
    })

    new_df["audio_fp"] = [
        make_audio_fp(species, id_)
        for species, id_ in zip(df["species"], df["id"])
    ]

    # remove bad paths / malformed species / missing files
    new_df = new_df[
        new_df["audio_fp"].apply(os.path.exists)
    ].copy()

    # optional: convert paths to strings
    new_df["audio_fp"] = new_df["audio_fp"].astype(str)
    new_df = new_df.reset_index(drop=True)

    parts = species.split()
    save_dir = os.path.join(DATASET_DIR, f'{'_'.join((parts[0][:3], parts[1]))}_multispecies_test')
    save_path = os.path.join(save_dir, 'test_info.csv')
    new_df.to_csv(save_path)